Step 12.1 — Import Libraries

In [1]:
import pandas as pd
import sqlite3
import os

Step 12.2 — Load Raw Dataset

In [2]:
df = pd.read_csv(
    "../data/raw/financial_fraud_detection_dataset.csv"
)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (5000, 14)


,Transaction_ID,Customer_ID,Transaction_Date,Transaction_Amount,Merchant_Category,Payment_Method,Device_Type,Location,Is_International,Previous_Transactions,Average_Spend,Account_Age_Days,Suspicious_Keyword,Fraudulent
0,T100000,CUST3252,04-10-2023 07:45,37.54,Travel,PayPal,POS,Bengaluru,0,94,417.40,1492,No,0
1,T100001,CUST1630,26-09-2023 08:29,240.81,Utilities,PayPal,Mobile,Bengaluru,0,76,335.47,66,No,0
2,T100002,CUST7852,18-07-2023 15:54,105.34,Entertainment,Debit Card,Mobile,Bengaluru,0,60,432.03,216,No,0
3,T100003,CUST4892,16-10-2023 17:10,73.04,Utilities,NetBanking,Desktop,Kolkata,0,68,488.30,449,No,0
4,T100004,CUST8831,09-09-2023 00:46,13.57,Utilities,Credit Card,Desktop,Bengaluru,0,57,437.03,754,No,0


Step 12.3 — Basic Verification

In [3]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Columns:
['Transaction_ID', 'Customer_ID', 'Transaction_Date', 'Transaction_Amount', 'Merchant_Category', 'Payment_Method', 'Device_Type', 'Location', 'Is_International', 'Previous_Transactions', 'Average_Spend', 'Account_Age_Days', 'Suspicious_Keyword', 'Fraudulent']

Missing values:
0

Duplicate rows:
0


Step 12.4 — Create Database Folder

In [4]:
os.makedirs(
    "../database",
    exist_ok=True
)

print("Database folder ready.")

Database folder ready.


Step 12.5 — Create SQLite Connection

In [5]:
connection = sqlite3.connect(
    "../database/fraud_detection.db"
)

print("SQLite database connected successfully.")

SQLite database connected successfully.


Step 12.6 — Write Data to SQLite

In [6]:
df.to_sql(
    "transactions",
    connection,
    if_exists="replace",
    index=False
)

print("Transactions table created successfully.")

Transactions table created successfully.


Step 12.7 — Verify Table

In [7]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """,
    connection
)

tables

,name
0,transactions


Step 12.8 — Check Number of Records

In [8]:
query = """
SELECT COUNT(*) AS total_transactions
FROM transactions;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,total_transactions
0,5000


Step 12.9 — Overall Fraud Summary

In [9]:
query = """
SELECT
    COUNT(*) AS total_transactions,
    SUM(Fraudulent) AS fraud_transactions,
    ROUND(
        100.0 * SUM(Fraudulent) / COUNT(*),
        2
    ) AS fraud_rate_percentage
FROM transactions;
"""

fraud_summary = pd.read_sql_query(
    query,
    connection
)

fraud_summary

,total_transactions,fraud_transactions,fraud_rate_percentage
0,5000,482,9.64


Step 12.10 — Fraud by Payment Method

In [10]:
query = """
SELECT
    Payment_Method,
    COUNT(*) AS total_transactions,
    SUM(Fraudulent) AS fraud_transactions,
    ROUND(
        100.0 * SUM(Fraudulent) / COUNT(*),
        2
    ) AS fraud_rate_percentage
FROM transactions
GROUP BY Payment_Method
ORDER BY fraud_rate_percentage DESC;
"""

fraud_by_payment = pd.read_sql_query(
    query,
    connection
)

fraud_by_payment

,Payment_Method,total_transactions,fraud_transactions,fraud_rate_percentage
0,UPI,961,100,10.41
1,Debit Card,1008,103,10.22
2,Credit Card,972,96,9.88
3,PayPal,1022,98,9.59
4,NetBanking,1037,85,8.20


Step 12.11 — Fraud by Merchant Category

In [11]:
query = """
SELECT
    Merchant_Category,
    COUNT(*) AS total_transactions,
    SUM(Fraudulent) AS fraud_transactions,
    ROUND(
        100.0 * SUM(Fraudulent) / COUNT(*),
        2
    ) AS fraud_rate_percentage
FROM transactions
GROUP BY Merchant_Category
ORDER BY fraud_rate_percentage DESC;
"""

fraud_by_merchant = pd.read_sql_query(
    query,
    connection
)

fraud_by_merchant

,Merchant_Category,total_transactions,fraud_transactions,fraud_rate_percentage
0,Entertainment,630,72,11.43
1,Food,684,74,10.82
2,Fashion,608,62,10.20
3,Health,606,61,10.07
4,Travel,599,56,9.35
5,Grocery,629,58,9.22
6,Utilities,594,50,8.42
7,Electronics,650,49,7.54


Step 12.12 — International vs Domestic

In [12]:
query = """
SELECT
    Is_International,
    COUNT(*) AS total_transactions,
    SUM(Fraudulent) AS fraud_transactions,
    ROUND(
        100.0 * SUM(Fraudulent) / COUNT(*),
        2
    ) AS fraud_rate_percentage
FROM transactions
GROUP BY Is_International
ORDER BY fraud_rate_percentage DESC;
"""

international_analysis = pd.read_sql_query(
    query,
    connection
)

international_analysis

,Is_International,total_transactions,fraud_transactions,fraud_rate_percentage
0,1,441,163,36.96
1,0,4559,319,7.00


Step 12.13 — Suspicious Keyword Analysis

In [13]:
query = """
SELECT
    Suspicious_Keyword,
    COUNT(*) AS total_transactions,
    SUM(Fraudulent) AS fraud_transactions,
    ROUND(
        100.0 * SUM(Fraudulent) / COUNT(*),
        2
    ) AS fraud_rate_percentage
FROM transactions
GROUP BY Suspicious_Keyword
ORDER BY fraud_rate_percentage DESC;
"""

keyword_analysis = pd.read_sql_query(
    query,
    connection
)

keyword_analysis

,Suspicious_Keyword,total_transactions,fraud_transactions,fraud_rate_percentage
0,Yes,260,123,47.31
1,No,4740,359,7.57


Step 12.14 — High-Value Transactions

In [14]:
query = """
SELECT
    Transaction_ID,
    Customer_ID,
    Transaction_Amount,
    Average_Spend,
    Fraudulent
FROM transactions
WHERE Transaction_Amount > 2 * Average_Spend
ORDER BY Transaction_Amount DESC
LIMIT 20;
"""

high_value_transactions = pd.read_sql_query(
    query,
    connection
)

high_value_transactions

,Transaction_ID,Customer_ID,Transaction_Amount,Average_Spend,Fraudulent
0,T100531,CUST9184,653.80,152.28,1
1,T102529,CUST5751,490.31,125.91,0
2,T103407,CUST6145,483.53,209.25,0
3,T103502,CUST8712,471.34,232.83,0
4,T103301,CUST5200,469.75,101.85,0
5,T101209,CUST4186,468.12,37.06,0
6,T100532,CUST6030,455.59,142.68,0
7,T103395,CUST8267,444.93,120.40,0
8,T101616,CUST7119,411.16,188.29,0
9,T102460,CUST4311,407.28,194.82,0


Step 12.15 — Fraudulent Transactions

In [15]:
query = """
SELECT
    Transaction_ID,
    Customer_ID,
    Transaction_Date,
    Transaction_Amount,
    Merchant_Category,
    Payment_Method,
    Is_International,
    Suspicious_Keyword,
    Fraudulent
FROM transactions
WHERE Fraudulent = 1
ORDER BY Transaction_Amount DESC;
"""

fraud_transactions = pd.read_sql_query(
    query,
    connection
)

fraud_transactions.head(20)

,Transaction_ID,Customer_ID,Transaction_Date,Transaction_Amount,Merchant_Category,Payment_Method,Is_International,Suspicious_Keyword,Fraudulent
0,T100531,CUST9184,25-07-2023 00:29,653.80,Grocery,PayPal,0,No,1
1,T103649,CUST1529,08-01-2024 13:07,617.88,Food,UPI,0,No,1
2,T101954,CUST9095,18-12-2023 03:30,512.44,Electronics,Debit Card,1,No,1
3,T104256,CUST5301,24-05-2023 02:26,487.72,Health,PayPal,0,No,1
4,T104559,CUST2951,23-07-2023 03:23,463.39,Electronics,Credit Card,0,No,1
5,T101715,CUST7322,15-10-2023 09:16,421.75,Travel,Credit Card,0,Yes,1
6,T102836,CUST6917,07-01-2024 01:31,399.92,Food,UPI,1,No,1
7,T104765,CUST1364,25-04-2023 04:15,387.63,Grocery,UPI,0,No,1
8,T104059,CUST3683,23-06-2023 01:18,385.89,Travel,NetBanking,1,No,1
9,T100261,CUST6250,15-05-2023 04:10,368.85,Food,Credit Card,0,No,1


Step 12.16 — Customer-Level Fraud Analysis

In [16]:
query = """
SELECT
    Customer_ID,
    COUNT(*) AS total_transactions,
    SUM(Fraudulent) AS fraud_transactions,
    ROUND(
        100.0 * SUM(Fraudulent) / COUNT(*),
        2
    ) AS customer_fraud_rate
FROM transactions
GROUP BY Customer_ID
HAVING SUM(Fraudulent) > 0
ORDER BY fraud_transactions DESC,
         customer_fraud_rate DESC;
"""

customer_fraud_analysis = pd.read_sql_query(
    query,
    connection
)

customer_fraud_analysis.head(20)

,Customer_ID,total_transactions,fraud_transactions,customer_fraud_rate
0,CUST2864,2,2,100.00
1,CUST3692,2,2,100.00
2,CUST4277,2,2,100.00
3,CUST7035,2,2,100.00
4,CUST7972,2,2,100.00
5,CUST8689,2,2,100.00
6,CUST9762,2,2,100.00
7,CUST2893,3,2,66.67
8,CUST5995,3,2,66.67
9,CUST8610,3,2,66.67


Step 12.17 — Fraud Amount Analysis

In [17]:
query = """
SELECT
    ROUND(SUM(Transaction_Amount), 2)
        AS total_transaction_amount,

    ROUND(
        SUM(
            CASE
                WHEN Fraudulent = 1
                THEN Transaction_Amount
                ELSE 0
            END
        ),
        2
    ) AS fraud_transaction_amount

FROM transactions;
"""

fraud_amount_analysis = pd.read_sql_query(
    query,
    connection
)

fraud_amount_analysis

,total_transaction_amount,fraud_transaction_amount
0,395903.32,39520.96


Step 12.18 — Fraud by Device Type

In [18]:

query = """
SELECT
    Device_Type,
    COUNT(*) AS total_transactions,
    SUM(Fraudulent) AS fraud_transactions,
    ROUND(
        100.0 * SUM(Fraudulent) / COUNT(*),
        2
    ) AS fraud_rate_percentage
FROM transactions
GROUP BY Device_Type
ORDER BY fraud_rate_percentage DESC;
"""

device_analysis = pd.read_sql_query(
    query,
    connection
)

device_analysis


,Device_Type,total_transactions,fraud_transactions,fraud_rate_percentage
0,Desktop,1679,172,10.24
1,POS,1673,159,9.50
2,Mobile,1648,151,9.16


Step 12.19 — Save SQL Results

In [19]:
os.makedirs(
    "../data/processed/sql_results",
    exist_ok=True
)

fraud_summary.to_csv(
    "../data/processed/sql_results/fraud_summary.csv",
    index=False
)

fraud_by_payment.to_csv(
    "../data/processed/sql_results/fraud_by_payment.csv",
    index=False
)

fraud_by_merchant.to_csv(
    "../data/processed/sql_results/fraud_by_merchant.csv",
    index=False
)

international_analysis.to_csv(
    "../data/processed/sql_results/international_analysis.csv",
    index=False
)

keyword_analysis.to_csv(
    "../data/processed/sql_results/keyword_analysis.csv",
    index=False
)

customer_fraud_analysis.to_csv(
    "../data/processed/sql_results/customer_fraud_analysis.csv",
    index=False
)

print("SQL analysis results saved successfully.")

SQL analysis results saved successfully.


Step 12.20 — Close Database

In [20]:
connection.close()

print("SQLite connection closed.")

SQLite connection closed.


## SQL Analytics Summary

The financial fraud dataset was loaded into a SQLite database and analyzed using SQL queries.

The analysis covered overall fraud statistics, fraud rates by payment method and merchant category, international versus domestic transactions, suspicious keyword patterns, high-value transactions, device-level fraud patterns, customer-level fraud activity, and the monetary value associated with fraudulent transactions.

The SQL outputs were exported as CSV files so they can be reused in the dashboard and reporting stages.